In [7]:
import pandas as pd
import re
import requests
from io import BytesIO
import numpy as np
from typing import Optional

In [9]:
file_id = "1zdZJxidp0DHzp0-MsvEMD7lxP3xsBwA3J4P9BAIvn0c"
sheet_name = "HIV2015"

url = (
    f"https://docs.google.com/spreadsheets/d/{file_id}/export"
    f"?format=xlsx&sheet={sheet_name}"
)

main_df = pd.read_excel(url, sheet_name=sheet_name)

In [11]:
new_header = main_df.iloc[1]
main_df = main_df.iloc[3:]
main_df.columns = new_header
main_df = main_df.reset_index(drop=True)

main_df.head()

1,Country,WHO Region,Population,Geographical Region,WHO Group,DALY,Adult DALYs,Children DALYs,Retention Rate,Retention Rate (ADULT),...,All ages,Children (0-14),Adults (15+),Year,NaN,http://apps.who.int/gho/data/node.main.626?lang=en,Estimated antiretroviral therapy coverage among people living with HIV (%),Reported number of people receiving antiretroviral therapy,Cleaned coverage,Cleaned number of people receiving antiretroviral therapy
0,Afghanistan,EMR,33736494,"East, South and South-East Asia",A,10752.548,9224.366,1528.182,72,73,...,92,100,92,2015,NaN,Afghanistan,5 [3-12],364,0.05,364
1,Albania,EUR,2880703,Europe and Central Asia,A,98.495261,96.59686,1.898401,92,92,...,92,77,92,2015,NaN,Albania,No data,423,NaN,423
2,Algeria,AFR,39871528,Middle East and North Africa,A,11586.0447,11055.12,530.9247,92,92,...,100,NaN,NaN,2015,NaN,Algeria,90 [70->95],7 915,0.9,7915
3,American Samoa,WPR,55537,NaN,A,28.518669,25.84525,2.673419,97.14,97.14,...,66,NaN,NaN,2015,NaN,Andorra,High-income country,No data,NaN,NaN
4,Andorra,EUR,78014,NaN,A,83.36974,83.20017,0.16957,97.14,97.14,...,85,100,85,2015,NaN,Angola,29 [20-40],90 204,0.29,90204


In [12]:
main_df.columns

Index([                                                                   'Country',
                                                                       'WHO Region',
                                                                       'Population',
                                                              'Geographical Region',
                                                                        'WHO Group',
                                                                             'DALY',
                                                                      'Adult DALYs',
                                                                   'Children DALYs',
                                                                   'Retention Rate',
                                                           'Retention Rate (ADULT)',
                                                           'Retention Rate (CHILD)',
                                                            '# Re

In [14]:
import re
import numpy as np
import pandas as pd
from typing import Optional



# Excel column letters - 0-based index

def _col_idx(col_letters: str) -> int:
    col_letters = col_letters.strip().upper()
    n = 0
    for ch in col_letters:
        n = n * 26 + (ord(ch) - ord("A") + 1)
    return n - 1



# Safe float conversion (handles %, blanks, commas)

def _to_float(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        s = x.strip().replace(",", "")
        if s == "" or s.lower() == "nan":
            return np.nan
        if s.endswith("%"):
            try:
                return float(s[:-1]) / 100.0
            except:
                return np.nan
        try:
            return float(s)
        except:
            return np.nan

    try:
        return float(x)
    except:
        return np.nan



# Extract drug token from column name: "Impact of 3TC" - "3TC"

def _drug_token_from_colname(colname) -> str:
    s = str(colname).strip()
    s = re.sub(r"(?i)^\s*impact\s+of\s+", "", s).strip()
    return s



# Check if regimen contains drug

def _contains_drug(regimen_text, drug_token: str) -> bool:
    if pd.isna(regimen_text):
        return False
    reg = str(regimen_text).upper().replace(" ", "")
    tok = str(drug_token).upper().replace(" ", "")
    return tok != "" and tok in reg



# Count number of drugs in regimen: "AZT + 3TC + NVP" - 3

def _regimen_size(regimen_text) -> float:
    if pd.isna(regimen_text):
        return 3.0

    parts = [p.strip() for p in re.split(r"\+", str(regimen_text)) if p.strip()]
    if len(parts) >= 2:
        return float(len(parts))

    return 3.0



# Impact formula block

def _impact_term(base, const, coef1, var, coef2, denom) -> float:

    if denom == 0 or pd.isna(denom):
        return 0.0

    if any(pd.isna(v) for v in [base, const, coef1, var, coef2]):
        return 0.0

    prod = const * coef1 * var * coef2
    d = 1.0 - prod

    if d == 0:
        return 0.0

    return (base * prod / d) / denom

def compute_impact_score_2015_avg(
    main_df: pd.DataFrame,
    drug_col_index: int,
    output_col: Optional[str] = None,
    excel_row0: int = 5,

    # stacked blocks
    first_line_excel_rows = range(8, 22),
    second_line_excel_rows = range(23, 40),

    regimen_col: str = "AS",
    debug: bool = False,
):

    def r(excel_row: int) -> int:
        return excel_row - excel_row0

    # Country input columns
    idx_G = _col_idx("G")   # Adult DALYs
    idx_H = _col_idx("H")   # Child DALYs
    idx_I = _col_idx("I")   # Retention
    idx_Q = _col_idx("Q")   # Adult coverage
    idx_T = _col_idx("T")   # Child coverage

    # Only averaged constants column
    idx_AP = _col_idx("AP")

    # regimen structure
    idx_REG = _col_idx(regimen_col)
    idx_ADULT_PROP = idx_REG + 1
    idx_ADULT_EFF  = idx_REG + 2
    idx_CHILD_PROP = idx_REG + 3
    idx_CHILD_EFF  = idx_REG + 4

    # constants from AP
    AP5  = _to_float(main_df.iat[r(5),  idx_AP])
    AP6  = _to_float(main_df.iat[r(6),  idx_AP])
    AP10 = _to_float(main_df.iat[r(10), idx_AP])
    AP11 = _to_float(main_df.iat[r(11), idx_AP])

    fl_rows = [r(x) for x in first_line_excel_rows if 0 <= r(x) < len(main_df)]
    sl_rows = [r(x) for x in second_line_excel_rows if 0 <= r(x) < len(main_df)]

    colname = main_df.columns[drug_col_index]
    drug_token = _drug_token_from_colname(colname)

    if output_col is None:
        output_col = f"Computed Impact Score ({drug_token})"

    scores = []

    for i in range(len(main_df)):

        row = main_df.iloc[i]

        G = _to_float(row.iloc[idx_G])
        H = _to_float(row.iloc[idx_H])
        Q = _to_float(row.iloc[idx_Q])
        T = _to_float(row.iloc[idx_T])
        I = _to_float(row.iloc[idx_I])

        total = 0.0

        # ---------------- FIRST LINE ----------------
        for rr in fl_rows:

            reg = main_df.iat[rr, idx_REG]

            if _contains_drug(reg, drug_token):

                denom = _regimen_size(reg)

                AT = _to_float(main_df.iat[rr, idx_ADULT_PROP])
                AU = _to_float(main_df.iat[rr, idx_ADULT_EFF])
                AV = _to_float(main_df.iat[rr, idx_CHILD_PROP])
                AW = _to_float(main_df.iat[rr, idx_CHILD_EFF])

                total += _impact_term(G, AP5,  AT, Q, AU, denom)
                total += _impact_term(H, AP10, AV, T, AW, denom)

        # ---------------- SECOND LINE ----------------
        for rr in sl_rows:

            reg = main_df.iat[rr, idx_REG]

            if _contains_drug(reg, drug_token):

                denom = _regimen_size(reg)

                AT = _to_float(main_df.iat[rr, idx_ADULT_PROP])
                AU = _to_float(main_df.iat[rr, idx_ADULT_EFF])
                AV = _to_float(main_df.iat[rr, idx_CHILD_PROP])
                AW = _to_float(main_df.iat[rr, idx_CHILD_EFF])

                total += _impact_term(G, AP6,  AT, Q, AU, denom)
                total += _impact_term(H, AP11, AV, T, AW, denom)

        # normalization
        try:
            norm = 100.0 / (100.0 - I)
            result = total / norm if norm != 0 else 0.0
        except:
            result = 0.0

        scores.append(result)

        if debug and i < 3:
            print(f"[DEBUG row {i}] total={total}, result={result}")

    main_df[output_col] = scores
    return main_df


In [ ]:
main_df = compute_impact_score_2015_avg(
    main_df,
    drug_col_index=24,   #3TC
    output_col="Computed Impact Score (3TC)",
    debug=True
)

main_df[["Country", "WHO Group", "3TC", "Computed Impact Score (3TC)"]].head(10)

[DEBUG row 0] total=105.09496470817135, result=29.426590118287976
[DEBUG row 1] total=0.0, result=0.0
[DEBUG row 2] total=2216.162706751355, result=177.2930165401084


1,Country,Country,WHO Group,3TC,Computed Impact Score (3TC)
0,Afghanistan,Albania,A,35.667802,29.426590
1,Albania,Algeria,A,0,0.000000
2,Algeria,Antigua and Barbuda,A,186.166832,177.293017
3,American Samoa,Argentina,A,0,0.049090
4,Andorra,Armenia,A,0.176927,0.158030
5,Angola,Azerbaijan,A,1085.331064,921.116066
6,Anguilla,Bahamas,B,0,0.000000
7,Antigua and Barbuda,Bahrain,B,0.826518,0.738241
8,Argentina,Bangladesh,B,3783.17524,3412.105196
9,Armenia,Barbados,A,5.906418,5.202796


In [17]:
impact_cols = list(
    zip(
        range(24, 35),
        main_df.columns[24:35]
    )
)


for idx, col in impact_cols:
    drug = _drug_token_from_colname(col)
    main_df = compute_impact_score_2015_avg(
        main_df,
        drug_col_index=idx,
        output_col=f"Computed Impact Score ({drug})",
        debug=False
    )

In [18]:
main_df.columns

Index([                                                                   'Country',
                                                                       'WHO Region',
                                                                       'Population',
                                                              'Geographical Region',
                                                                        'WHO Group',
                                                                             'DALY',
                                                                      'Adult DALYs',
                                                                   'Children DALYs',
                                                                   'Retention Rate',
                                                           'Retention Rate (ADULT)',
                                                           'Retention Rate (CHILD)',
                                                            '# Re

In [19]:
import numpy as np
import pandas as pd

# --- your computed drug impact columns ---
computed_drug_cols_wanted = [
    'Computed Impact Score (3TC)',
    'Computed Impact Score (ABC)',
    'Computed Impact Score (AZT)',
    'Computed Impact Score (ddl)',
    'Computed Impact Score (d4T)',
    'Computed Impact Score (EFV)',
    'Computed Impact Score (FTC)',
    'Computed Impact Score (LPV/r)',
    'Computed Impact Score (NVP)',
    'Computed Impact Score (TDF)',
    'Computed Impact Score (ATV/r)',
]

# only keep the ones that exist right now
computed_drug_cols = [c for c in computed_drug_cols_wanted if c in main_df.columns]
missing = [c for c in computed_drug_cols_wanted if c not in main_df.columns]
if missing:
    print("⚠️ Missing computed columns (not included in overall):")
    for m in missing:
        print("   -", m)

# ensure numeric
for c in computed_drug_cols:
    main_df[c] = pd.to_numeric(main_df[c], errors="coerce")

# per-country overall computed impact
main_df["Computed Overall Treatment Impact"] = main_df[computed_drug_cols].sum(axis=1, skipna=True)


In [20]:
global_computed_total = main_df["Computed Overall Treatment Impact"].sum(skipna=True)
global_actual_total = pd.to_numeric(main_df["Overall Treatment Impact"], errors="coerce").sum(skipna=True)

print("Global totals (sum across countries):")
print("  Computed:", global_computed_total)
print("  Actual  :", global_actual_total)
print("  Diff    :", global_computed_total - global_actual_total)


Global totals (sum across countries):
  Computed: 1887890.485889323
  Actual  : 2639736.7135998914
  Diff    : -751846.2277105683


In [22]:
from IPython.display import display

pairs = [
    ("3TC",   "3TC",   "Computed Impact Score (3TC)"),
    ("ABC",   "ABC",   "Computed Impact Score (ABC)"),
    ("AZT",   "AZT",   "Computed Impact Score (AZT)"),
    ("ddl",   "ddl",   "Computed Impact Score (ddl)"),
    ("d4T",   "d4T",   "Computed Impact Score (d4T)"),
    ("EFV",   "EFV",   "Computed Impact Score (EFV)"),
    ("FTC",   "FTC",   "Computed Impact Score (FTC)"),
    ("LPV/r", "LPV/r", "Computed Impact Score (LPV/r)"),
    ("NVP",   "NVP",   "Computed Impact Score (NVP)"),
    ("TDF",   "TDF",   "Computed Impact Score (TDF)"),
    ("ATV/r", "ATV/r", "Computed Impact Score (ATV/r)"),
    ("Overall", "Overall Treatment Impact", "Computed Overall Treatment Impact"),
]

# Keep only pairs where both columns exist
pairs_existing = []
for drug, a, c in pairs:
    if a in main_df.columns and c in main_df.columns:
        pairs_existing.append((drug, a, c))
    else:
        print(f"⚠️ Skipping {drug}: missing column(s):",
              [x for x in [a, c] if x not in main_df.columns])

# Make sure numeric
for _, a, c in pairs_existing:
    main_df[a] = pd.to_numeric(main_df[a], errors="coerce")
    main_df[c] = pd.to_numeric(main_df[c], errors="coerce")

# Build comparison table
out = main_df[["Country", "WHO Group"]].copy()

for drug, a, c in pairs_existing:
    out[f"{drug} | Actual"] = main_df[a]
    out[f"{drug} | Computed"] = main_df[c]
    out[f"{drug} | Diff (Comp-Act)"] = main_df[c] - main_df[a]

display(out.head(50))


1,Country,Country,WHO Group,3TC | Actual,3TC | Computed,3TC | Diff (Comp-Act),ABC | Actual,ABC | Computed,ABC | Diff (Comp-Act),AZT | Actual,...,NVP | Diff (Comp-Act),TDF | Actual,TDF | Computed,TDF | Diff (Comp-Act),ATV/r | Actual,ATV/r | Computed,ATV/r | Diff (Comp-Act),Overall | Actual,Overall | Computed,Overall | Diff (Comp-Act)
0,Afghanistan,Albania,A,35.667802,29.426590,-6.241212,3.483114,3.483114,4.071801e-10,15.831563,...,-11.583309,23.749159,22.376962,-1.372197,0.412628,0.412628,-8.447915e-11,129.229982,89.667641,-39.562341
1,Albania,Algeria,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
2,Algeria,Antigua and Barbuda,A,186.166832,177.293017,-8.873816,3.714239,3.714239,-3.088858e-10,70.801802,...,-50.968654,164.992819,159.265709,-5.727110,2.513475,2.513475,-2.715987e-10,710.727413,538.743199,-171.984214
3,American Samoa,Argentina,A,0.000000,0.049090,0.049090,0.000000,0.000219,2.192879e-04,0.000000,...,0.008266,0.000000,0.044577,0.044577,0.000000,0.000863,8.625692e-04,0.000000,0.149536,0.149536
4,Andorra,Armenia,A,0.176927,0.158030,-0.018897,0.000706,0.000706,5.924807e-14,0.065802,...,-0.045639,0.165287,0.143502,-0.021785,0.002777,0.002777,-7.630225e-14,0.685301,0.481382,-0.203919
5,Angola,Azerbaijan,A,1085.331064,921.116066,-164.214998,84.823245,84.823245,-1.729319e-09,466.427030,...,-340.591233,780.409167,734.116996,-46.292170,13.092495,13.092495,-5.313836e-09,3984.248983,2805.010812,-1179.238171
6,Anguilla,Bahamas,B,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
7,Antigua and Barbuda,Bahrain,B,0.826518,0.738241,-0.088277,0.003298,0.003298,-2.455805e-14,0.307398,...,-0.213205,0.772143,0.670375,-0.101769,0.012972,0.012972,4.984676e-12,3.201400,2.248787,-0.952612
8,Argentina,Bangladesh,B,3783.175240,3412.105196,-371.070044,122.256802,122.256802,-3.853155e-08,1502.665107,...,-1087.761868,3192.017671,2983.225778,-208.791893,50.927850,50.927850,2.701455e-09,14313.300080,10379.378772,-3933.921308
9,Armenia,Barbados,A,5.906418,5.202796,-0.703622,0.023988,0.023988,-5.980869e-12,2.206146,...,-1.524520,5.511532,4.712557,-0.798975,0.094334,0.094334,-1.913650e-11,22.882402,15.856107,-7.026295


In [24]:
pd.set_option('display.max_rows', None)
main_df

1,Country,WHO Region,Population,Geographical Region,WHO Group,DALY,Adult DALYs,Children DALYs,Retention Rate,Retention Rate (ADULT),...,Computed Impact Score (AZT),Computed Impact Score (ddl),Computed Impact Score (d4T),Computed Impact Score (EFV),Computed Impact Score (FTC),Computed Impact Score (LPV/r),Computed Impact Score (NVP),Computed Impact Score (TDF),Computed Impact Score (ATV/r),Computed Overall Treatment Impact
0,Afghanistan,EMR,33736494,"East, South and South-East Asia",A,10752.548,9224.366,1528.182,72,73,...,3.703817,0.102642,0.523836,4.606680,0.267710,19.921067,4.842593,22.376962,0.412628,89.667641
1,Albania,EUR,2880703,Europe and Central Asia,A,98.495261,96.59686,1.898401,92,92,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Algeria,AFR,39871528,Middle East and North Africa,A,11586.0447,11055.12,530.9247,92,92,...,17.294537,0.088267,0.455925,17.449507,1.631119,132.913992,26.123412,159.265709,2.513475,538.743199
3,American Samoa,WPR,55537,NaN,A,28.518669,25.84525,2.673419,97.14,97.14,...,0.005439,0.000000,0.000000,0.005157,0.000560,0.035364,0.008266,0.044577,0.000863,0.149536
4,Andorra,EUR,78014,NaN,A,83.36974,83.20017,0.16957,97.14,97.14,...,0.017510,0.000000,0.000000,0.016601,0.001802,0.113844,0.026610,0.143502,0.002777,0.481382
5,Angola,AFR,27859305,Sub-Saharan Africa,A,648951.4,485015.9,163935.5,97.14,97.14,...,109.720265,2.455045,12.552467,130.603869,8.494919,639.469697,148.565748,734.116996,13.092495,2805.010812
6,Anguilla,AMR,14723,NaN,B,0,0,0,97.14,97.14,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,Antigua and Barbuda,AMR,99923,NaN,B,413.85098,388.6716,25.17938,97.14,97.14,...,0.081800,0.000000,0.000000,0.077550,0.008417,0.531826,0.124310,0.670375,0.012972,2.248787
8,Argentina,AMR,43417765,Latin America and the Caribbean,B,78326.475,73886.85,4439.625,66,97.14,...,361.541607,3.136901,16.203024,379.610383,33.047185,2486.473994,530.850050,2983.225778,50.927850,10379.378772
9,Armenia,EUR,2916950,Europe and Central Asia,A,776.945553,773.9614,2.984153,85,85,...,0.591556,0.000000,0.000000,0.560640,0.061207,3.713744,0.895285,4.712557,0.094334,15.856107


In [26]:
main_df.to_csv("impact_score.csv", index=False)
print("File saved as impact_score.csv")

File saved as impact_score.csv
